In [6]:
from tool_server.utils.utils import *
from tool_server.utils.prompts import tool_planning_model_prompt_one_tool_call
from tqdm import tqdm
import json, os
from copy import deepcopy
metric_dict = {
    "chartqa": ["average_llm_score"],
    "chartgemma": ["average_llm_score"],
    "charxiv": ["average_descriptive_score", "average_reasoning_score"],
    "docvqa": ["average_llm_score"],
    "ocrbench": ["average_llm_score"],
    "reachqa": ["average_llm_score"],
    "vstar": ["average_direct_attributes_score","average_relative_position_score"],
}

model_name_dict = {
    "qwen25vl3b": "Qwen-2.5VL-3B /w Tools",
    "qwen25vl3b_sftv1": "Qwen-2.5VL-3B SFT v1",
    "qwen25vl3b_notool": "Qwen-2.5VL-3B /wo Tools",
    "qwen25vl7b": "Qwen-2.5VL-7B /w Tools",
    "qwen25vl7b_sftv1": "Qwen-2.5VL-7B SFT v1",
    "qwen25vl7b_notool": "Qwen-2.5VL-7B /wo Tools",
}

task_list = ["chartqa", "chartgemma", "charxiv", "docvqa", "ocrbench", "reachqa","vstar"]
res_base_dir = "/mnt/petrelfs/songmingyang/code/reasoning/tool-agent/tool_server/tf_eval/scripts/logs/results"


In [7]:

def get_res_dict(model_name_dict, res_base_dir = "/mnt/petrelfs/songmingyang/code/reasoning/tool-agent/tool_server/tf_eval/scripts/logs/results"):
    res_dict = {}
    for model_name,full_name in model_name_dict.items():
        result_path = os.path.join(res_base_dir, model_name, "qwen25_allres_llm.jsonl")
        result_data = process_jsonl(result_path)
        result_dict = {}
        for result_item in result_data:
            task_name = result_item["task_name"]
            result_dict[task_name] = result_item
        res_dict[full_name] = result_dict
    return res_dict
        
def print_res_table(big_res_dict, task_list, metric_dict=metric_dict):
    res = ""
    for model_name, res_dict in big_res_dict.items():
        res += f"\\textbf{{{model_name}}}"
        
        for task_name in task_list:
            if task_name in res_dict:
                result_item = res_dict[task_name]
                if "task_name" not in result_item:
                    print(f"Warning: task_name not found in result_item for {model_name} and {task_name}")
                acc = get_result_number(result_item)
                for acc_idx, acc in enumerate(acc):
                    # all_res = [get_result_number(v.get(task_name,{}))[acc_idx] for k,v in res_dict.items()]
                    # all_res.sort()
                    all_res = [-1,-2]
                    if acc == all_res[-1]:
                        res += f" & \\textbf{acc*100:.2f}" 
                    elif acc == all_res[-2]:
                        res += f" & \\underline{acc*100:.2f}" 
                    else:
                        res += f" & {acc*100:.2f}"
            else:
                metric_num = len(metric_dict[task_name])
                for _ in range(metric_num):
                    res += " & -"
        res += " \\\\ \n"
    return res

def get_result_number(result_item, metric_dict=metric_dict):
    task_name = result_item["task_name"]
    metrics = metric_dict[task_name]
    return [result_item["llm_eval_summary"][i] for i in metrics]
   
            

In [8]:
res_dict = get_res_dict(model_name_dict, res_base_dir)


In [10]:
res = print_res_table(res_dict, task_list)
print(res)

\textbf{Qwen-2.5VL-3B /w Tools} & 62.60 & 34.71 & 34.83 & 26.30 & 72.35 & 69.60 & 31.40 & 29.57 & 48.68 \\ 
\textbf{Qwen-2.5VL-3B SFT v1} & 60.72 & 32.39 & 32.47 & 20.60 & 52.27 & 42.40 & 27.10 & 23.48 & 40.79 \\ 
\textbf{Qwen-2.5VL-3B /wo Tools} & 56.68 & 35.41 & 29.40 & 26.00 & 77.27 & 70.80 & 32.55 & 43.48 & 57.89 \\ 
\textbf{Qwen-2.5VL-7B /w Tools} & 72.08 & 47.89 & 60.77 & 33.60 & 88.45 & 72.40 & 44.95 & 45.22 & 63.16 \\ 
\textbf{Qwen-2.5VL-7B SFT v1} & 67.72 & 40.14 & 49.70 & 25.30 & 59.43 & 52.00 & 34.55 & 31.30 & 50.00 \\ 
\textbf{Qwen-2.5VL-7B /wo Tools} & 76.88 & 50.60 & 71.20 & 37.60 & 91.61 & 82.50 & 37.35 & 64.35 & 68.42 \\ 



In [5]:
metric_dict

{'chartqa': ['Exact_Match_Acc'],
 'chartgemma': ['Acc'],
 'charxiv': ['type_res/descriptive', 'type_res/reasoning'],
 'docvqa': ['exact_score'],
 'ocrbench': ['Acc'],
 'reachqa': ['chart_type_res/AVG'],
 'vstar': ['category_res/direct_attributes', 'category_res/relative_position']}